In [2]:
import pandas as pd
import numpy as np

# Path to the original Excel file
file_path = "supply_chain_dataset.xlsx"

# Load the main dataset
df = pd.read_excel(
    file_path,
    sheet_name="supply_chain_dataset1"
)

# Display the first 5 rows
df.head()

,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.95,20.48,0,0,18.63
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.95,20.48,1,0,39.62
3,2024-01-04,SKU_1,WH_1,SUP_8,South,24,516,14,379,0,13.95,20.48,0,0,19.43
4,2024-01-05,SKU_1,WH_1,SUP_8,West,21,495,14,379,0,13.95,20.48,0,0,18.70


In [4]:
df.columns.tolist()

['Date',
 'SKU_ID',
 'Warehouse_ID',
 'Supplier_ID',
 'Region',
 'Units_Sold',
 'Inventory_Level',
 'Supplier_Lead_Time_Days',
 'Reorder_Point',
 'Order_Quantity',
 'Unit_Cost',
 'Unit_Price',
 'Promotion_Flag',
 'Stockout_Flag',
 'Demand_Forecast']

In [5]:
# Standardize column names to lowercase snake_case

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# Display the standardized column names
df.columns.tolist()

['date',
 'sku_id',
 'warehouse_id',
 'supplier_id',
 'region',
 'units_sold',
 'inventory_level',
 'supplier_lead_time_days',
 'reorder_point',
 'order_quantity',
 'unit_cost',
 'unit_price',
 'promotion_flag',
 'stockout_flag',
 'demand_forecast']

In [6]:
# Verify data types after standardization

print("Data types:")
print(df.dtypes)

Data types:
date                       datetime64[ns]
sku_id                             object
warehouse_id                       object
supplier_id                        object
region                             object
units_sold                          int64
inventory_level                     int64
supplier_lead_time_days             int64
reorder_point                       int64
order_quantity                      int64
unit_cost                         float64
unit_price                        float64
promotion_flag                      int64
stockout_flag                       int64
demand_forecast                   float64
dtype: object


In [7]:
# Check the data types of key columns

print("Date data type:")
print(df["date"].dtype)

print("\nNumerical data types:")
print(df[[
    "units_sold",
    "inventory_level",
    "supplier_lead_time_days",
    "reorder_point",
    "order_quantity",
    "unit_cost",
    "unit_price",
    "demand_forecast"
]].dtypes)

print("\nFlag data types:")
print(df[[
    "promotion_flag",
    "stockout_flag"
]].dtypes)

Date data type:
datetime64[ns]

Numerical data types:
units_sold                   int64
inventory_level              int64
supplier_lead_time_days      int64
reorder_point                int64
order_quantity               int64
unit_cost                  float64
unit_price                 float64
demand_forecast            float64
dtype: object

Flag data types:
promotion_flag    int64
stockout_flag     int64
dtype: object


In [8]:
# Check categorical columns for leading/trailing spaces

categorical_columns = [
    "sku_id",
    "warehouse_id",
    "supplier_id",
    "region"
]

for column in categorical_columns:
    whitespace_count = (
        df[column].astype(str) != df[column].astype(str).str.strip()
    ).sum()
    
    print(f"{column}: {whitespace_count} values with extra spaces")

sku_id: 0 values with extra spaces
warehouse_id: 0 values with extra spaces
supplier_id: 0 values with extra spaces
region: 0 values with extra spaces


In [9]:
# Display unique values for categorical columns

for column in categorical_columns:
    print(f"\n{column}:")
    print(sorted(df[column].unique()))


sku_id:
['SKU_1', 'SKU_10', 'SKU_11', 'SKU_12', 'SKU_13', 'SKU_14', 'SKU_15', 'SKU_16', 'SKU_17', 'SKU_18', 'SKU_19', 'SKU_2', 'SKU_20', 'SKU_21', 'SKU_22', 'SKU_23', 'SKU_24', 'SKU_25', 'SKU_26', 'SKU_27', 'SKU_28', 'SKU_29', 'SKU_3', 'SKU_30', 'SKU_31', 'SKU_32', 'SKU_33', 'SKU_34', 'SKU_35', 'SKU_36', 'SKU_37', 'SKU_38', 'SKU_39', 'SKU_4', 'SKU_40', 'SKU_41', 'SKU_42', 'SKU_43', 'SKU_44', 'SKU_45', 'SKU_46', 'SKU_47', 'SKU_48', 'SKU_49', 'SKU_5', 'SKU_50', 'SKU_6', 'SKU_7', 'SKU_8', 'SKU_9']

warehouse_id:
['WH_1', 'WH_2', 'WH_3', 'WH_4', 'WH_5']

supplier_id:
['SUP_1', 'SUP_10', 'SUP_2', 'SUP_3', 'SUP_4', 'SUP_5', 'SUP_6', 'SUP_7', 'SUP_8', 'SUP_9']

region:
['East', 'North', 'South', 'West']


In [10]:
# Create a summary of categorical columns

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique_Values": [df[column].nunique() for column in categorical_columns],
    "Null_Count": [df[column].isnull().sum() for column in categorical_columns]
})

categorical_summary

,Column,Unique_Values,Null_Count
0,sku_id,50,0
1,warehouse_id,5,0
2,supplier_id,10,0
3,region,4,0


In [12]:
# Review the statistical range of numerical columns


numeric_columns = df.select_dtypes(include="number").columns.tolist()

numerical_summary = df[numeric_columns].describe().T

numerical_summary

,count,mean,std,min,25%,50%,75%,max
units_sold,91250.0,20.054564,9.068602,0.00,13.00,20.00,27.00,59.00
inventory_level,91250.0,471.522312,133.488002,168.00,370.00,461.00,564.00,990.00
supplier_lead_time_days,91250.0,7.984000,3.907929,2.00,4.00,8.00,11.00,14.00
reorder_point,91250.0,300.068000,54.879945,201.00,252.00,300.00,346.00,398.00
order_quantity,91250.0,19.272493,82.340831,0.00,0.00,0.00,0.00,499.00
unit_cost,91250.0,12.203320,4.574982,5.02,8.18,11.99,16.32,19.76
unit_price,91250.0,18.261800,7.121136,6.95,12.00,18.18,23.39,35.10
promotion_flag,91250.0,0.101589,0.302109,0.00,0.00,0.00,0.00,1.00
stockout_flag,91250.0,0.000000,0.000000,0.00,0.00,0.00,0.00,0.00
demand_forecast,91250.0,20.082033,9.503955,0.00,12.95,19.95,26.93,61.42


In [13]:
# Investigate order quantity distribution

order_quantity_summary = df["order_quantity"].describe()

print(order_quantity_summary)

count    91250.000000
mean        19.272493
std         82.340831
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        499.000000
Name: order_quantity, dtype: float64


In [ ]:
# Count records with and without replenishment orders

zero_orders = (df["order_quantity"] == 0).sum()
positive_orders = (df["order_quantity"] > 0).sum()

print(f"Records with no order       : {zero_orders:,}")
print(f"Records with an order       : {positive_orders:,}")
print(f"Total records               : {len(df):,}")

Records with no order       : 86,223
Records with an order       : 5,027
Total records               : 91,250


In [15]:
# Check whether selling price is greater than unit cost

price_cost_check = (df["unit_price"] >= df["unit_cost"]).sum()

print(
    f"Records where unit price >= unit cost: "
    f"{price_cost_check:,} / {len(df):,}"
)

Records where unit price >= unit cost: 91,250 / 91,250


In [17]:
# Check business-rule violations for key supply chain fields

business_rule_checks = {
    "Negative / zero inventory": (df["inventory_level"] <= 0).sum(),
    "Invalid lead time": (df["supplier_lead_time_days"] <= 0).sum(),
    "Invalid reorder point": (df["reorder_point"] <= 0).sum(),
    "Negative demand": (df["units_sold"] < 0).sum(),
    "Negative demand forecast": (df["demand_forecast"] < 0).sum()
}

print("Business rule violations:")
for check, count in business_rule_checks.items():
    print(f"{check}: {count:,}")

Business rule violations:
Negative / zero inventory: 0
Invalid lead time: 0
Invalid reorder point: 0
Negative demand: 0
Negative demand forecast: 0


In [18]:
# Check replenishment behavior against the reorder point

order_records = df[df["order_quantity"] > 0]

orders_at_or_below_reorder_point = (
    order_records["inventory_level"] <= order_records["reorder_point"]
).sum()

orders_above_reorder_point = (
    order_records["inventory_level"] > order_records["reorder_point"]
).sum()

print(f"Records with replenishment orders      : {len(order_records):,}")
print(f"Orders at/below reorder point          : {orders_at_or_below_reorder_point:,}")
print(f"Orders above reorder point             : {orders_above_reorder_point:,}")

Records with replenishment orders      : 5,027
Orders at/below reorder point          : 0
Orders above reorder point             : 5,027


In [19]:
# Check the relationship between actual demand and forecast

df["forecast_error"] = df["units_sold"] - df["demand_forecast"]

print("Forecast error statistics:")
print(df["forecast_error"].describe())

Forecast error statistics:
count    91250.000000
mean        -0.027469
std          2.987263
min        -14.080000
25%         -2.030000
50%         -0.010000
75%          1.990000
max         13.000000
Name: forecast_error, dtype: float64


In [20]:
# Calculate estimated demand during supplier lead time

df["lead_time_demand"] = (
    df["units_sold"] * df["supplier_lead_time_days"]
)

# Compare lead-time demand with the reorder point

df["reorder_point_gap"] = (
    df["reorder_point"] - df["lead_time_demand"]
)

df[[
    "units_sold",
    "supplier_lead_time_days",
    "lead_time_demand",
    "reorder_point",
    "reorder_point_gap"
]].head()

,units_sold,supplier_lead_time_days,lead_time_demand,reorder_point,reorder_point_gap
0,10,14,140,379,239
1,17,14,238,379,141
2,35,14,490,379,-111
3,24,14,336,379,43
4,21,14,294,379,85


In [21]:
# Summarize the relationship between reorder point and lead-time demand

gap_summary = {
    "Reorder point above lead-time demand": (
        df["reorder_point_gap"] > 0
    ).sum(),

    "Reorder point equal to lead-time demand": (
        df["reorder_point_gap"] == 0
    ).sum(),

    "Reorder point below lead-time demand": (
        df["reorder_point_gap"] < 0
    ).sum()
}

for check, count in gap_summary.items():
    print(f"{check}: {count:,}")

Reorder point above lead-time demand: 78,258
Reorder point equal to lead-time demand: 207
Reorder point below lead-time demand: 12,785


In [22]:
# Check whether each SKU + Warehouse + Date combination is unique

grain_columns = [
    "date",
    "sku_id",
    "warehouse_id"
]

duplicate_grain_count = df.duplicated(
    subset=grain_columns
).sum()

print(
    f"Duplicate Date + SKU + Warehouse combinations: "
    f"{duplicate_grain_count:,}"
)

Duplicate Date + SKU + Warehouse combinations: 0


In [23]:
# Check the number of daily observations for each SKU + Warehouse combination

sku_warehouse_counts = (
    df.groupby(["sku_id", "warehouse_id"])["date"]
    .nunique()
)

print("Minimum number of dates:", sku_warehouse_counts.min())
print("Maximum number of dates:", sku_warehouse_counts.max())
print("Expected number of daily dates:", df["date"].nunique())

Minimum number of dates: 365
Maximum number of dates: 365
Expected number of daily dates: 365


In [24]:
# Create analytical date features

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["quarter"] = df["date"].dt.quarter
df["week"] = df["date"].dt.isocalendar().week
df["day_of_week"] = df["date"].dt.dayofweek
df["day_name"] = df["date"].dt.day_name()

# Preview the new date features
df[
    [
        "date",
        "year",
        "month",
        "month_name",
        "quarter",
        "week",
        "day_of_week",
        "day_name"
    ]
].head()

,date,year,month,month_name,quarter,week,day_of_week,day_name
0,2024-01-01,2024,1,January,1,1,0,Monday
1,2024-01-02,2024,1,January,1,1,1,Tuesday
2,2024-01-03,2024,1,January,1,1,2,Wednesday
3,2024-01-04,2024,1,January,1,1,3,Thursday
4,2024-01-05,2024,1,January,1,1,4,Friday


In [25]:
# Calculate inventory value

df["inventory_value"] = (
    df["inventory_level"] * df["unit_cost"]
)

df[[
    "sku_id",
    "warehouse_id",
    "inventory_level",
    "unit_cost",
    "inventory_value"
]].head()

,sku_id,warehouse_id,inventory_level,unit_cost,inventory_value
0,SKU_1,WH_1,592,13.95,8258.40
1,SKU_1,WH_1,575,13.95,8021.25
2,SKU_1,WH_1,540,13.95,7533.00
3,SKU_1,WH_1,516,13.95,7198.20
4,SKU_1,WH_1,495,13.95,6905.25


In [26]:
# Calculate forecast error

df["forecast_error"] = (
    df["units_sold"] - df["demand_forecast"]
)

# Calculate absolute forecast error

df["absolute_forecast_error"] = (
    df["forecast_error"].abs()
)

df[[
    "date",
    "sku_id",
    "units_sold",
    "demand_forecast",
    "forecast_error",
    "absolute_forecast_error"
]].head()

,date,sku_id,units_sold,demand_forecast,forecast_error,absolute_forecast_error
0,2024-01-01,SKU_1,10,8.52,1.48,1.48
1,2024-01-02,SKU_1,17,18.63,-1.63,1.63
2,2024-01-03,SKU_1,35,39.62,-4.62,4.62
3,2024-01-04,SKU_1,24,19.43,4.57,4.57
4,2024-01-05,SKU_1,21,18.70,2.30,2.30


In [27]:
# Calculate inventory coverage in days

df["inventory_coverage_days"] = np.where(
    df["units_sold"] > 0,
    df["inventory_level"] / df["units_sold"],
    np.nan
)

df[[
    "date",
    "sku_id",
    "warehouse_id",
    "units_sold",
    "inventory_level",
    "inventory_coverage_days"
]].head()

,date,sku_id,warehouse_id,units_sold,inventory_level,inventory_coverage_days
0,2024-01-01,SKU_1,WH_1,10,592,59.200000
1,2024-01-02,SKU_1,WH_1,17,575,33.823529
2,2024-01-03,SKU_1,WH_1,35,540,15.428571
3,2024-01-04,SKU_1,WH_1,24,516,21.500000
4,2024-01-05,SKU_1,WH_1,21,495,23.571429


In [28]:
# Calculate inventory position relative to reorder point

df["inventory_reorder_gap"] = (
    df["inventory_level"] - df["reorder_point"]
)

df[[
    "date",
    "sku_id",
    "warehouse_id",
    "inventory_level",
    "reorder_point",
    "inventory_reorder_gap"
]].head()

,date,sku_id,warehouse_id,inventory_level,reorder_point,inventory_reorder_gap
0,2024-01-01,SKU_1,WH_1,592,379,213
1,2024-01-02,SKU_1,WH_1,575,379,196
2,2024-01-03,SKU_1,WH_1,540,379,161
3,2024-01-04,SKU_1,WH_1,516,379,137
4,2024-01-05,SKU_1,WH_1,495,379,116


In [29]:
# Calculate forecast error percentage

df["forecast_error_pct"] = np.where(
    df["units_sold"] > 0,
    (df["forecast_error"] / df["units_sold"]) * 100,
    np.nan
)

df[[
    "date",
    "sku_id",
    "units_sold",
    "demand_forecast",
    "forecast_error",
    "forecast_error_pct"
]].head()

,date,sku_id,units_sold,demand_forecast,forecast_error,forecast_error_pct
0,2024-01-01,SKU_1,10,8.52,1.48,14.800000
1,2024-01-02,SKU_1,17,18.63,-1.63,-9.588235
2,2024-01-03,SKU_1,35,39.62,-4.62,-13.200000
3,2024-01-04,SKU_1,24,19.43,4.57,19.041667
4,2024-01-05,SKU_1,21,18.70,2.30,10.952381


In [30]:
# Calculate estimated demand during supplier lead time

df["lead_time_demand"] = (
    df["units_sold"] * df["supplier_lead_time_days"]
)

df[[
    "sku_id",
    "warehouse_id",
    "units_sold",
    "supplier_lead_time_days",
    "lead_time_demand"
]].head()

,sku_id,warehouse_id,units_sold,supplier_lead_time_days,lead_time_demand
0,SKU_1,WH_1,10,14,140
1,SKU_1,WH_1,17,14,238
2,SKU_1,WH_1,35,14,490
3,SKU_1,WH_1,24,14,336
4,SKU_1,WH_1,21,14,294


In [31]:
# Calculate potential sales value of current inventory

df["inventory_sales_value"] = (
    df["inventory_level"] * df["unit_price"]
)

df[[
    "sku_id",
    "warehouse_id",
    "inventory_level",
    "unit_price",
    "inventory_sales_value"
]].head()

,sku_id,warehouse_id,inventory_level,unit_price,inventory_sales_value
0,SKU_1,WH_1,592,20.48,12124.16
1,SKU_1,WH_1,575,20.48,11776.00
2,SKU_1,WH_1,540,20.48,11059.20
3,SKU_1,WH_1,516,20.48,10567.68
4,SKU_1,WH_1,495,20.48,10137.60


In [32]:
# Create a reorder status based on current inventory

df["reorder_status"] = np.select(
    [
        df["inventory_level"] < df["reorder_point"],
        df["inventory_level"] == df["reorder_point"],
        df["inventory_level"] > df["reorder_point"]
    ],
    [
        "Below Reorder Point",
        "At Reorder Point",
        "Above Reorder Point"
    ],
    default="Unknown"
)

# Preview the result
df[[
    "sku_id",
    "warehouse_id",
    "inventory_level",
    "reorder_point",
    "inventory_reorder_gap",
    "reorder_status"
]].head()

,sku_id,warehouse_id,inventory_level,reorder_point,inventory_reorder_gap,reorder_status
0,SKU_1,WH_1,592,379,213,Above Reorder Point
1,SKU_1,WH_1,575,379,196,Above Reorder Point
2,SKU_1,WH_1,540,379,161,Above Reorder Point
3,SKU_1,WH_1,516,379,137,Above Reorder Point
4,SKU_1,WH_1,495,379,116,Above Reorder Point


In [33]:
# Calculate unit margin and margin percentage

df["unit_margin"] = (
    df["unit_price"] - df["unit_cost"]
)

df["margin_pct"] = (
    df["unit_margin"] / df["unit_price"]
) * 100

df[[
    "sku_id",
    "unit_cost",
    "unit_price",
    "unit_margin",
    "margin_pct"
]].head()

,sku_id,unit_cost,unit_price,unit_margin,margin_pct
0,SKU_1,13.95,20.48,6.53,31.884766
1,SKU_1,13.95,20.48,6.53,31.884766
2,SKU_1,13.95,20.48,6.53,31.884766
3,SKU_1,13.95,20.48,6.53,31.884766
4,SKU_1,13.95,20.48,6.53,31.884766


In [35]:
# Final validation of the cleaned dataset

print("Dataset shape:")
print(df.shape)

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nTotal duplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape:
(91250, 34)

Total missing values:
1240

Total duplicate rows:
0

Data types:
date                       datetime64[ns]
sku_id                             object
warehouse_id                       object
supplier_id                        object
region                             object
units_sold                          int64
inventory_level                     int64
supplier_lead_time_days             int64
reorder_point                       int64
order_quantity                      int64
unit_cost                         float64
unit_price                        float64
promotion_flag                      int64
stockout_flag                       int64
demand_forecast                   float64
forecast_error                    float64
lead_time_demand                    int64
reorder_point_gap                   int64
year                                int32
month                               int32
month_name                         object
quarter                  

In [36]:
# Revalidate the business grain after data preparation

grain_duplicate_count = df.duplicated(
    subset=["date", "sku_id", "warehouse_id"]
).sum()

print(
    "Duplicate Date + SKU + Warehouse combinations:",
    grain_duplicate_count
)

Duplicate Date + SKU + Warehouse combinations: 0


In [37]:
# Identify where the missing values were introduced

missing_by_column = df.isnull().sum()

missing_by_column = missing_by_column[
    missing_by_column > 0
].sort_values(ascending=False)

print("Columns containing missing values:")
print(missing_by_column)

Columns containing missing values:
inventory_coverage_days    620
forecast_error_pct         620
dtype: int64


In [38]:
# Check the number of records with zero actual demand

zero_demand_count = (df["units_sold"] == 0).sum()

print(f"Records with zero units sold: {zero_demand_count:,}")

Records with zero units sold: 620


In [39]:
# Check derived numerical fields for infinite or invalid values

derived_numeric_columns = [
    "inventory_value",
    "inventory_sales_value",
    "forecast_error",
    "absolute_forecast_error",
    "forecast_error_pct",
    "inventory_coverage_days",
    "inventory_reorder_gap",
    "lead_time_demand",
    "unit_margin",
    "margin_pct"
]

infinite_counts = np.isinf(
    df[derived_numeric_columns]
).sum()

print("Infinite values by derived column:")
print(infinite_counts)

print(
    "\nTotal infinite values:",
    infinite_counts.sum()
)

Infinite values by derived column:
inventory_value            0
inventory_sales_value      0
forecast_error             0
absolute_forecast_error    0
forecast_error_pct         0
inventory_coverage_days    0
inventory_reorder_gap      0
lead_time_demand           0
unit_margin                0
margin_pct                 0
dtype: int64

Total infinite values: 0


In [40]:
# Export the cleaned dataset

output_file = "cleaned_supply_chain_dataset.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Cleaned dataset exported successfully: {output_file}")

Cleaned dataset exported successfully: cleaned_supply_chain_dataset.csv
